# Clean the Netflix Dataset

This notebook inspects and cleans the Kaggle Netflix Movies and TV Shows dataset. It keeps the original CSV unchanged, explains decisions for missing values, converts mixed-type fields, and writes a separate cleaned CSV.

**Dataset:** [Netflix Movies and TV Shows on Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)

Before running the notebook, download `netflix_titles.csv` from Kaggle and place it in `data/netflix_titles.csv` beside this notebook.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from netflix_cleaner import clean_netflix_dataframe

DATA_PATH = Path("data/netflix_titles.csv")
OUTPUT_PATH = Path("data/cleaned_netflix_titles.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download netflix_titles.csv from Kaggle "
        "and place it in the data/ folder."
    )

raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw_df):,} rows and {len(raw_df.columns)} columns.")

## Inspect the source data

Start by checking example rows, column types, summary statistics, and missingness before changing anything.

In [ ]:
display(raw_df.head())
raw_df.info()
display(raw_df.describe(include="all").T)

In [ ]:
normalized_for_missing = raw_df.copy()
for column in normalized_for_missing.select_dtypes(include=["object", "string"]).columns:
    normalized_for_missing[column] = normalized_for_missing[column].replace(
        r"^\s*$", pd.NA, regex=True
    )

missing_before = (
    normalized_for_missing.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda table: (table["missing_count"] / len(raw_df) * 100).round(2))
    .sort_values("missing_count", ascending=False)
)
display(missing_before)

## Cleaning choices

- Blank strings are treated as missing after trimming whitespace.
- Duplicate `show_id` records keep their first occurrence. Rows without `show_id`, `title`, or `type` are dropped because these fields identify the record.
- Missing `director`, `cast`, `country`, and `rating` values become `Unknown`; a missing date or invalid numeric value stays empty rather than being guessed.
- `date_added` becomes a datetime column and `release_year` becomes a nullable integer.
- `duration` contains unlike units (minutes and seasons), so it is split into a numeric `duration_value` and a `duration_unit` column. Unexpected non-empty formats are labeled `Unknown` for review.

In [ ]:
rows_before = len(raw_df)
duplicate_ids = raw_df["show_id"].duplicated().sum() if "show_id" in raw_df else 0

cleaned_df = clean_netflix_dataframe(raw_df)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Repeated show_id values in source: {duplicate_ids:,}")
print(f"Rows after cleaning: {len(cleaned_df):,}")
display(cleaned_df.head())

## Review the cleaned data

Check the updated types, remaining missing values, duplicate identifiers, and duration units before exporting.

In [ ]:
cleaned_df.info()
display(cleaned_df.isna().sum().rename("missing_count").to_frame())
print("Duplicate show_id values after cleaning:", cleaned_df["show_id"].duplicated().sum())
if "duration_unit" in cleaned_df:
    display(cleaned_df["duration_unit"].value_counts(dropna=False).rename("rows").to_frame())

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d", encoding="utf-8")
print(f"Saved cleaned data to {OUTPUT_PATH}")